# Analysis of the final out-of-fold predictions — the three challenges

The organisers name three difficulties of the task. This notebook measures how the final model handles each of them,
using only the outputs of `pipeline.ipynb` (out-of-fold probabilities and component candidates) and the training
annotations; it needs no GPU.

1. **Fine-structure fidelity** — thin filaments and barbs. *Performance by filament size; boundary complexity of
   matched GT / prediction pairs; zoomed examples.*
2. **Filament–background discrimination** — separating filament material from chromospheric noise. *Error budget of
   misses and false positives; human–human agreement as a reference; calibration of the component scorer.*
3. **Instance coherence** — one coherent instance instead of fragments or merged neighbours. *Fragmentation and
   over-merging counts; an oracle ceiling for merging fragments; the closing-radius trade-off.*

Every number is out of fold (each image predicted by the model that did not see it) and uses the official
per-annotator-record PQ.

In [ ]:
import os, sys, json, pickle, glob
from pathlib import Path

REPO_DIR = Path(globals().get("REPO_DIR", ".."))
DATA_DIR = Path(globals().get("DATA_DIR", os.environ.get("DATA_DIR", "/kaggle/input")))
WORK_DIR = Path(globals().get("WORK_DIR", os.environ.get("WORK_DIR", "./work")))   # outputs of pipeline.ipynb
CACHE_DIR = Path(globals().get("CACHE_DIR", WORK_DIR / "cache"))
PROB_PATH = Path(globals().get("PROB_PATH", WORK_DIR / "oof" / "oof_prob.npy"))
OUT_DIR = Path(globals().get("OUT_DIR", WORK_DIR / "analysis"))
WORKERS = int(globals().get("WORKERS", max(1, min(8, os.cpu_count() or 2))))
DRY = os.environ.get("DRY") == "1"

sys.path.insert(0, str(REPO_DIR))
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from filament.config import load_config, scaled
from filament.data import Annotations, build_cache, find_files, read_cache
from filament import pipeline as pl, evaluate as ev, postprocess as pp, analysis as an

cfg = load_config(REPO_DIR / "configs" / "final.yaml")
if DRY:
    cfg = scaled(cfg, 256)
OUT_DIR.mkdir(parents=True, exist_ok=True)
log = pl.log_to(OUT_DIR / "analysis.log")
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a8a"
SEED = cfg["seed"]

def ci(b):
    return f"{b['diff']:+.4f} [{b['lo']:+.4f}, {b['hi']:+.4f}]"

In [ ]:
# ---------------------------------------------------------------------------------- inputs and baseline check
ann = Annotations(find_files(DATA_DIR, "*kaggle2026_train.json")[0],
                  [d for d in find_files(DATA_DIR, "train_images") if os.path.isdir(d)][0])
with open(WORK_DIR / "oof_candidates.pkl", "rb") as fh:
    K = pickle.load(fh)
C, STEMS, FOLD_OF = K["C"], K["stems"], K["fold_of"]
order = pd.read_csv(PROB_PATH.parent / "oof_order.csv").stem.tolist()
assert order == STEMS, "oof_prob.npy and oof_candidates.pkl have a different image order"
build_cache(ann, STEMS, CACHE_DIR, cfg, workers=WORKERS, log=log)        # network inputs (features, figures)
R = pl.evaluate_scorer(C, STEMS, FOLD_OF, cfg, SEED)
U, TAU = R["U"], R["tau"]
KEEP = {s: pp.select(U[s], TAU) for s in STEMS}
ALL = {s: np.ones(C[s]["n"], bool) for s in STEMS}
base = ev.summary(R["P"])
log(f"baseline OOF PQ {base['PQ']:.4f} (tau {TAU:.4f}) — should equal results.json")
T = {}                                                                     # every table, saved at the end

In [ ]:
P = an.image_pass(PROB_PATH, STEMS, ann, C, U, TAU, cfg, workers=WORKERS, log=log)
GA = {s: P[s]["gt_area"] for s in STEMS}
FN = pd.DataFrame([r for s in STEMS for r in P[s]["fn"]])
FP = pd.DataFrame([r for s in STEMS for r in P[s]["fp"]])
PAIRS = pd.DataFrame([r for s in STEMS for r in P[s]["pairs"]])
log(f"{len(FN)} misses, {len(FP)} false positives, {len(PAIRS)} matched pairs (per record)")

## Challenge 1 — fine-structure fidelity
**Size strata.** GT filaments are split into area quartiles. True-positive pairs and misses are counted in the GT's
bin, false positives in the prediction's bin, so the bins add up exactly to the overall PQ. Comparing *recall* with
*SQ* (mean IoU of the matches) separates "not found" from "found but poorly delineated".

In [ ]:
edges = np.quantile(np.concatenate([a for s in STEMS for a in GA[s] if len(a)]), [0.25, 0.5, 0.75])
names = [f"<{edges[0]:.0f}", f"{edges[0]:.0f}–{edges[1]:.0f}", f"{edges[1]:.0f}–{edges[2]:.0f}", f">{edges[2]:.0f}"]
SB = {"scorer": an.size_parts(C, STEMS, KEEP, GA, edges), "all components": an.size_parts(C, STEMS, ALL, GA, edges)}
assert abs(ev.pq(SB["scorer"].sum(1)) - base["PQ"]) < 1e-9
rows = []
for b, nm in enumerate(names):
    for v, A in SB.items():
        s, tp, fp, fn = A[:, b].sum(0)
        rows.append(dict(size_px=nm, selection=v, PQ=ev.pq(A[:, b]), recall=tp / max(tp + fn, 1), SQ=s / max(tp, 1),
                         TP=int(tp), FP=int(fp), FN=int(fn)))
    rows[-2]["scorer_gain"] = ci(ev.paired_bootstrap(SB["scorer"][:, b], SB["all components"][:, b]))
T["c1_size_strata"] = pd.DataFrame(rows)
log(T["c1_size_strata"].round(4).to_string(index=False))
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
x = np.arange(len(names))
for j, (col, title) in enumerate((("PQ", "PQ"), ("recall", "recall (detection)"), ("SQ", "SQ (IoU of matches)"), ("FP", "false positives"))):
    for k, (v, colr) in enumerate((("all components", GREY), ("scorer", BLUE))):
        y = T["c1_size_strata"].query("selection == @v")[col].values
        ax[j].bar(x + (k - 0.5) * 0.38, y, 0.36, color=colr, label=v)
    ax[j].set_xticks(x); ax[j].set_xticklabels(names, fontsize=8); ax[j].set_title(title, fontsize=10)
    ax[j].set_xlabel("prediction area (px)" if col == "FP" else "GT area (px)", fontsize=8)
    ax[j].spines[["top", "right"]].set_visible(False); ax[j].grid(axis="y", alpha=0.25); ax[j].set_axisbelow(True)
ax[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "c1_size_strata.png", dpi=140); plt.show()

**Boundary complexity of matched pairs.** For every true-positive pair we compare the outline of the annotation and
of the prediction: isoperimetric complexity P²/(4πA) (1 for a disc, large for ragged or barbed outlines) and the
number of skeleton end points (each barb adds one). If barbs were lost, predictions would be systematically simpler
than their annotations, and IoU would fall with GT complexity.

In [ ]:
pr = PAIRS.dropna(subset=["gt_complexity", "pred_complexity"]).copy()
pr["complexity_ratio"] = pr.pred_complexity / pr.gt_complexity
pr["gt_complexity_q"] = pd.qcut(pr.gt_complexity, 4, labels=["Q1 simple", "Q2", "Q3", "Q4 complex"])
from scipy.stats import spearmanr
rho_c = spearmanr(pr.gt_complexity, pr.pred_complexity).correlation
rho_e = spearmanr(pr.gt_ends, pr.pred_ends).correlation
T["c1_boundary"] = pr.groupby("gt_complexity_q", observed=True).agg(
    n=("iou", "size"), iou_mean=("iou", "mean"), gt_complexity=("gt_complexity", "median"),
    pred_complexity=("pred_complexity", "median"), gt_ends=("gt_ends", "median"), pred_ends=("pred_ends", "median")).reset_index()
C1 = dict(pairs=len(pr), complexity_ratio_median=float(pr.complexity_ratio.median()), spearman_complexity=float(rho_c),
          spearman_endpoints=float(rho_e), endpoints_gt_median=float(pr.gt_ends.median()),
          endpoints_pred_median=float(pr.pred_ends.median()),
          share_pred_simpler=float((pr.pred_complexity < pr.gt_complexity).mean()))
log(T["c1_boundary"].round(3).to_string(index=False)); log(json.dumps(C1))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(pr.gt_complexity, pr.pred_complexity, s=4, alpha=0.25, color=BLUE)
lim = [1, max(pr.gt_complexity.quantile(0.99), pr.pred_complexity.quantile(0.99))]
ax[0].plot(lim, lim, color=GREY, lw=1, ls="--"); ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("GT complexity P²/(4πA)"); ax[0].set_ylabel("prediction complexity")
ax[0].set_title(f"matched pairs (n={len(pr)}), Spearman {rho_c:.2f}", fontsize=10)
ax[1].bar(range(4), T["c1_boundary"].iou_mean, color=BLUE)
ax[1].set_xticks(range(4)); ax[1].set_xticklabels(T["c1_boundary"].gt_complexity_q, fontsize=8)
ax[1].set_ylabel("mean IoU of match"); ax[1].set_title("IoU by GT boundary complexity", fontsize=10)
for a in ax: a.spines[["top", "right"]].set_visible(False); a.grid(alpha=0.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(OUT_DIR / "c1_boundary_complexity.png", dpi=140); plt.show()

In [ ]:
# zoomed examples: the six matched pairs with the most barb-like GT outlines (most skeleton end points)
full = cfg["data"]["full"]
ex = pr.sort_values(["gt_ends", "gt_area"], ascending=False).drop_duplicates("stem").head(6)
O = np.load(PROB_PATH, mmap_mode="r"); IDX = {s: i for i, s in enumerate(STEMS)}
fig, axs = plt.subplots(2, 3, figsize=(13, 8.6))
for a, r in zip(axs.ravel(), ex.itertuples()):
    lab, _ = pp.components_cfg(O[IDX[r.stem]].astype(np.float32) / 255.0, cfg)
    g = np.zeros(full * full, bool); g[ann.record_masks(r.stem, full)[r.record][r.gt]] = True; g = g.reshape(full, full)
    y0, y1, x0, x1 = r.bbox; pad = 40
    y0, x0, y1, x1 = max(y0 - pad, 0), max(x0 - pad, 0), min(y1 + pad, full), min(x1 + pad, full)
    img = read_cache(CACHE_DIR, "x", r.stem, full)[y0:y1, x0:x1]
    a.imshow(img, cmap="gray")
    a.contour(g[y0:y1, x0:x1], levels=[0.5], colors=[AQUA], linewidths=2.2)
    a.contour(lab[y0:y1, x0:x1] == r.pred + 1, levels=[0.5], colors=[ORANGE], linewidths=1.0)
    a.set_title(f"{r.stem}  IoU {r.iou:.2f}  ends GT {r.gt_ends} / pred {r.pred_ends}", fontsize=8); a.axis("off")
fig.suptitle("annotation (aqua) vs prediction (orange): barb-rich filaments", fontsize=11)
plt.tight_layout(); plt.savefig(OUT_DIR / "c1_barb_examples.png", dpi=140); plt.show()

## Challenge 2 — filament–background discrimination
**Error budget.** Every missed annotation (FN) is classified as *not detected* (no component touches it),
*low overlap* (touched, but no component reaches IoU 0.5: fragmented or wrongly sized) or *removed by the scorer*
(a matching component existed but was discarded). On images with several annotator records we also check whether
the missed filament was drawn by that annotator only. Every false positive is *background* (touches no annotation of
any record), *partial overlap*, or *a true positive for another annotator* of the same image.

In [ ]:
fn_t = FN.groupby("category").agg(n=("area", "size"), median_area=("area", "median")).reset_index()
fn_t["share"] = fn_t.n / fn_t.n.sum()
multi = FN.dropna(subset=["only_this_annotator"])
C2 = dict(fn_total=len(FN), fn_multi_record=len(multi),
          fn_drawn_by_one_annotator_only=float(multi.only_this_annotator.astype(bool).mean()) if len(multi) else None)
fp_t = FP.groupby("category").agg(n=("area", "size"), median_area=("area", "median")).reset_index()
fp_t["share"] = fp_t.n / fp_t.n.sum()
T["c2_misses"], T["c2_false_positives"] = fn_t, fp_t
log(fn_t.round(3).to_string(index=False)); log(fp_t.round(3).to_string(index=False)); log(json.dumps(C2))

In [ ]:
# human-human agreement (every ordered pair of annotator records on the same image) vs the model on the same images
H = np.array([h for s in STEMS for h in P[s]["human"]], float)
multi_stems = [s for s in STEMS if len(ann.records(s)) > 1]
sel = np.array([s in set(multi_stems) for s in STEMS])
T["c2_human_vs_model"] = pd.DataFrame([dict(who="annotator vs annotator", images=len(multi_stems), **ev.summary(H)),
                                       dict(who="model (OOF)", images=len(multi_stems), **ev.summary(R["P"][sel])),
                                       dict(who="model (OOF), all images", images=len(STEMS), **base)])
log(T["c2_human_vs_model"].round(4).to_string(index=False))

**Scorer.** The scorer predicts *u*, a component's expected matched IoU over the image's records. A component
improves PQ when *u* > PQ/2, so a calibrated scorer makes the keep threshold a consequence of the metric rather
than a tuned constant. Below: reliability of *u*, the theoretical threshold vs the tuned one, and feature ablations.

In [ ]:
uu = np.concatenate([U[s] for s in STEMS]); yy = np.concatenate([C[s]["Y"] for s in STEMS])
cal, ece = an.calibration(uu, yy, bins=10)
T["c2_calibration"] = pd.DataFrame(cal)
tau_theory = base["PQ"] / 2
P_theory = pl.parts(C, STEMS, U, tau_theory)
abl = {"tuned tau": R, "no scorer (all components)": dict(P=R["P_all"], tau=None)}
abl["19 base features only"] = pl.evaluate_scorer(C, STEMS, FOLD_OF, cfg, SEED, extra=False)
abl["+ station letter feature"] = pl.evaluate_scorer(C, STEMS, FOLD_OF, cfg, SEED, station=True)
rows = [dict(variant=f"theoretical tau = PQ/2 = {tau_theory:.3f}", PQ=ev.pq(P_theory), vs_final=ci(ev.paired_bootstrap(P_theory, R["P"])))]
for k, v in abl.items():
    rows.append(dict(variant=k + (f" ({v['tau']:.3f})" if v["tau"] is not None else ""), PQ=ev.pq(v["P"]),
                     vs_final=ci(ev.paired_bootstrap(v["P"], R["P"])) if k != "tuned tau" else "–"))
T["c2_scorer"] = pd.DataFrame(rows)
log(f"scorer ECE {ece:.4f}\n" + T["c2_scorer"].to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot([0, 1], [0, 1], color=GREY, lw=1, ls="--")
ax[0].plot(T["c2_calibration"].u_mean, T["c2_calibration"].y_mean, marker="o", color=BLUE)
ax[0].axvline(TAU, color=ORANGE, lw=1, label=f"keep threshold τ={TAU:.3f}")
ax[0].set_xlabel("predicted u (OOF)"); ax[0].set_ylabel("realised matched IoU"); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title(f"scorer reliability (ECE {ece:.3f})", fontsize=10)
t_, p_ = zip(*R["curve"])
ax[1].plot(t_, p_, color=BLUE); ax[1].axvline(TAU, color=ORANGE, lw=1, label="tuned")
ax[1].axvline(tau_theory, color=GREY, lw=1, ls="--", label="PQ/2"); ax[1].legend(frameon=False, fontsize=8)
ax[1].set_xlabel("keep threshold τ"); ax[1].set_ylabel("OOF PQ"); ax[1].set_title("threshold: theory vs tuned", fontsize=10)
for a in ax: a.spines[["top", "right"]].set_visible(False); a.grid(alpha=0.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(OUT_DIR / "c2_scorer.png", dpi=140); plt.show()

## Challenge 3 — instance coherence
**Fragmentation and over-merging** are counted per record like the official one-to-many statistics: a GT overlapped
by two or more predictions is fragmented, a prediction overlapping two or more GT has merged neighbours. The
**merge oracle** merges, with knowledge of the ground truth, the fragments of every unmatched GT when their union
would match it — an upper bound on what any fragment-merging step could gain. The **closing radius** of the
post-processing directly trades fragmentation against over-merging.

In [ ]:
deg = an.overlap_degrees(C, STEMS, KEEP)
Pm, n_merged = an.merge_oracle(C, STEMS, KEEP, GA)
C3 = dict(**deg, merge_oracle_pq=ev.pq(Pm), merge_oracle_gain=ev.pq(Pm) - base["PQ"], merge_oracle_new_matches=n_merged)
log(json.dumps(C3))
rows = [dict(close_radius=cfg["postprocess"]["close"], PQ=base["PQ"], vs_final="–", **deg)]
for r_ in ([2] if DRY else [3, 9]):
    c_ = json.loads(json.dumps(cfg)); c_["postprocess"]["close"] = r_
    Cr = pl.candidates(PROB_PATH, STEMS, ann, CACHE_DIR, c_, workers=WORKERS, log=log)
    Rr = pl.evaluate_scorer(Cr, STEMS, FOLD_OF, c_, SEED)
    kr = {s: pp.select(Rr["U"][s], Rr["tau"]) for s in STEMS}
    rows.append(dict(close_radius=r_, PQ=ev.pq(Rr["P"]), vs_final=ci(ev.paired_bootstrap(Rr["P"], R["P"])),
                     **an.overlap_degrees(Cr, STEMS, kr)))
T["c3_closing"] = pd.DataFrame(rows).sort_values("close_radius")
log(T["c3_closing"].to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
tc = T["c3_closing"]
ax[0].plot(tc.close_radius, tc.PQ, marker="o", color=BLUE); ax[0].set_xlabel("closing radius (px)"); ax[0].set_ylabel("OOF PQ")
ax[1].plot(tc.close_radius, tc.gt_fragmented, marker="o", color=ORANGE, label="GT split into ≥2 predictions")
ax[1].plot(tc.close_radius, tc.pred_merged, marker="s", color=AQUA, label="prediction covering ≥2 GT")
ax[1].set_xlabel("closing radius (px)"); ax[1].set_ylabel("count (per record)"); ax[1].legend(frameon=False, fontsize=8)
for a in ax: a.spines[["top", "right"]].set_visible(False); a.grid(alpha=0.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(OUT_DIR / "c3_closing_tradeoff.png", dpi=140); plt.show()

In [ ]:
# --------------------------------------------------------------------------------------------------- save
for k, v in T.items():
    v.to_csv(OUT_DIR / f"{k}.csv", index=False)
PAIRS.to_csv(OUT_DIR / "matched_pairs.csv", index=False)
(OUT_DIR / "analysis_summary.json").write_text(json.dumps(dict(baseline=base, tau=TAU, size_edges=edges.tolist(),
    challenge1=C1, challenge2=dict(**C2, scorer_ece=ece, tau_theory=tau_theory), challenge3=C3), indent=1, default=float))
log(f"saved -> {OUT_DIR}")

## Example segmentations (report figure)

In [ ]:
# Example segmentations for the report (template asks for "a few examples of the segmentations").
# Needs only the variables of the "inputs and baseline check" cell (ann, C, STEMS, KEEP, cfg, PROB_PATH, OUT_DIR);
# the image pass is not required.
# Three single-annotator images with >= 6 filaments, chosen by rule (not by eye): per-image PQ at the 75th, 50th and
# 10th percentiles. Background: the limb-flattened image (without CLAHE), downsampled for display.
import cv2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from filament import evaluate as ev, postprocess as pp
from filament.preprocess import find_disk, flatten

full = cfg["data"]["full"]
O = np.load(PROB_PATH, mmap_mode="r"); IDX = {s: i for i, s in enumerate(STEMS)}

rows = []
for s in STEMS:
    mats = C[s]["mats"]
    if len(mats) != 1 or mats[0].shape[0] == 0:            # single-annotator images only: one unambiguous outline set
        continue
    S, tp, fp, fn = ev.record_score(mats[0], KEEP[s])
    gt_area = np.array([len(g) for g in ann.record_masks(s, full)[0]], float) * (2048.0 / full) ** 2
    rows.append(dict(stem=s, n_gt=mats[0].shape[0], pq=S / max(tp + 0.5 * fp + 0.5 * fn, 1e-9),
                     tp=tp, fp=fp, fn=fn, med_area=float(np.median(gt_area))))
D = pd.DataFrame(rows)

def at_quantile(df, q):
    t = df.pq.quantile(q)
    return df.iloc[(df.pq - t).abs().argsort().iloc[0]]

D6 = D[D.n_gt >= 6]
assert len(D6) >= 4, "too few single-annotator images with >= 6 filaments"
picks = [("good", at_quantile(D6, 0.75)), ("median", at_quantile(D6, 0.50)), ("difficult", at_quantile(D6, 0.10))]
for name, r in picks:
    print(f"{name:16s} {r.stem}  PQ {r.pq:.3f}  GT {r.n_gt}  TP {r.tp}  FP {r.fp}  FN {r.fn}  median GT area {r.med_area:.0f}")

AQUA, ORANGE, YELLOW = "#1baf7a", "#eb6834", "#f2c84b"
fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.55))
for a, (name, r) in zip(axs, picks):
    s = r.stem
    raw = cv2.imread(ann.meta.set_index("stem").at[s, "path"], cv2.IMREAD_GRAYSCALE)
    if raw.shape[0] != full:
        raw = cv2.resize(raw, (full, full), interpolation=cv2.INTER_AREA)
    pc = cfg["preprocess"]
    cy, cx, rr = find_disk(raw, pc["disk_radius_prior"], pc["disk_radius_tol"])
    img, _ = flatten(raw, cy, cx, rr, nbins=pc["flat_bins"], clip=tuple(pc["flat_clip"]))
    lab, n = pp.components_cfg(O[IDX[s]].astype(np.float32) / 255.0, cfg)
    assert n == C[s]["n"], f"{s}: components differ from the candidate file"
    keep = KEEP[s]
    kept = np.isin(lab, np.flatnonzero(keep) + 1)
    dropped = np.isin(lab, np.flatnonzero(~keep) + 1)
    g = np.zeros(full * full, bool)
    for idx in ann.record_masks(s, full)[0]:
        g[idx] = True
    g = g.reshape(full, full)
    ys, xs = np.nonzero(img > 0)                          # crop to the solar disk
    sl = (slice(ys.min(), ys.max() + 1), slice(xs.min(), xs.max() + 1))
    crop = img[sl]
    h, w = crop.shape
    disp = cv2.resize(crop, (int(w * 800 / h), 800), interpolation=cv2.INTER_AREA)
    a.imshow(disp, cmap="gray", vmin=0.0, vmax=1.0, extent=(-0.5, w - 0.5, h - 0.5, -0.5))
    a.contour(g[sl], levels=[0.5], colors=[AQUA], linewidths=1.1)
    a.contour(kept[sl], levels=[0.5], colors=[ORANGE], linewidths=0.6)
    if dropped.any():
        a.contour(dropped[sl], levels=[0.5], colors=[YELLOW], linewidths=0.5, linestyles="dashed")
    a.set_title(f"{name}: PQ {r.pq:.2f}", fontsize=8)
    a.axis("off")
handles = [Line2D([], [], color=AQUA, lw=1.4, label="annotation"),
           Line2D([], [], color=ORANGE, lw=1.0, label="submitted prediction"),
           Line2D([], [], color=YELLOW, lw=0.8, ls="--", label="removed by scorer")]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=7, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.06, 1, 1), w_pad=0.4)
plt.savefig(OUT_DIR / "examples.pdf", bbox_inches="tight")
plt.savefig(OUT_DIR / "examples.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved:", OUT_DIR / "examples.pdf", "and", OUT_DIR / "examples.png")